# Calibration Analysis

Loads E2's saved test predictions and computes:
- **Reliability diagram** (predicted vs observed sepsis rate per probability bin)
- **Brier score** (mean squared error between predicted prob and true label)
- **Expected Calibration Error (ECE)** (weighted average of bin-level miscalibration)

No retraining needed — just analyses existing predictions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

PROJECT_PATH = '/content/drive/MyDrive/Sepsis'
MODEL_PATH = f'{PROJECT_PATH}/models/E2'
OUTPUT_PATH = f'{PROJECT_PATH}/models/calibration'
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Load E2 test predictions
data = np.load(f'{MODEL_PATH}/test_predictions.npz')
y_prob = data['preds']
y_true = data['labels']
print(f'Loaded {len(y_prob):,} test predictions')
print(f'Positive rate: {y_true.mean()*100:.1f}%')

In [ ]:
# Brier score (lower is better; perfect = 0, random = 0.25 for balanced)
brier = np.mean((y_prob - y_true) ** 2)

# Expected Calibration Error (ECE) with 10 bins
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
ece = 0
bin_stats = []
for i in range(n_bins):
    mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i+1] if i < n_bins-1 else y_prob <= bin_edges[i+1])
    if mask.sum() == 0:
        continue
    avg_prob = y_prob[mask].mean()
    avg_true = y_true[mask].mean()
    weight = mask.sum() / len(y_prob)
    ece += weight * abs(avg_prob - avg_true)
    bin_stats.append({
        'bin_low': float(bin_edges[i]), 'bin_high': float(bin_edges[i+1]),
        'count': int(mask.sum()), 'avg_predicted': float(avg_prob),
        'observed_positive_rate': float(avg_true)
    })

print(f'Brier score: {brier:.4f}')
print(f'  (Lower is better; random = {y_true.mean()*(1-y_true.mean()):.4f})')
print(f'Expected Calibration Error: {ece:.4f}')
print(f'  (Lower is better; perfect = 0, well-calibrated typically <0.05)')

print(f'\nBin-by-bin breakdown:')
print(f'{"Bin":<15}{"N":>10}{"Avg Pred":>12}{"Observed":>12}{"Diff":>10}')
for b in bin_stats:
    diff = b['avg_predicted'] - b['observed_positive_rate']
    print(f'  [{b["bin_low"]:.1f}-{b["bin_high"]:.1f}]  {b["count"]:>10,}  {b["avg_predicted"]:>11.3f}  {b["observed_positive_rate"]:>11.3f}  {diff:>+9.3f}')

In [ ]:
# Reliability diagram
prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='uniform')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Reliability diagram
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
ax.plot(prob_pred, prob_true, 'o-', color='#1f77b4', linewidth=2, markersize=10,
        label=f'Multi-Agent E2\nBrier={brier:.4f}, ECE={ece:.4f}')
ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Observed Positive Rate', fontsize=12)
ax.set_title('Reliability Diagram', fontsize=13)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(fontsize=11, loc='upper left')
ax.grid(alpha=0.3)

# Histogram of predicted probabilities
ax = axes[1]
ax.hist(y_prob[y_true == 0], bins=50, alpha=0.6, label='Sepsis-negative', color='#4ecdc4')
ax.hist(y_prob[y_true == 1], bins=50, alpha=0.6, label='Sepsis-positive', color='#ff6b6b')
ax.set_xlabel('Predicted Probability', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Prediction Distribution by True Label', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/calibration_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_PATH}/calibration_plot.png')

In [ ]:
# Save metrics
results = {
    'brier_score': float(brier),
    'expected_calibration_error': float(ece),
    'n_bins': n_bins,
    'bin_stats': bin_stats,
    'n_predictions': int(len(y_prob)),
    'positive_rate': float(y_true.mean())
}
with open(f'{OUTPUT_PATH}/calibration_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved calibration_results.json')

print('\nINTERPRETATION FOR THESIS:')
if ece < 0.05:
    print(f'  ECE = {ece:.4f} — well-calibrated. Predictions can be trusted as probabilities.')
elif ece < 0.10:
    print(f'  ECE = {ece:.4f} — moderate miscalibration. Predictions are useful for ranking but should be re-calibrated (Platt scaling, isotonic regression) before deployment.')
else:
    print(f'  ECE = {ece:.4f} — substantial miscalibration. Re-calibration required for deployment.')